## feature slection, engineering

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn import tree
import numpy as np


#데이터셋 불러오기
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 고윳값 Drop
X_train=train_df.drop(["index","name","health_status"],axis=1)
X_test=test_df.drop(["index","name"],axis=1)
y_train=train_df['health_status']

# 주요 피처 리스트
important_features = ['iron_lvl', 'stature_inch', 'fat_content', 'age_enzyme_ratio', 'Total_Cholesterol']

# 결과 확인
X_test


In [ ]:
def create_interaction_features(df):
    df['bmi'] = df['mass_kg'] / ((df['stature_inch'] + 100) / 100) ** 2
    df['waist_to_age_ratio'] = df['midsection_inch'] / 2024-df['birth']
    df['iron_per_stature'] = df['iron_lvl'] / (df['stature_inch'] + 1e-6)
    df['bmi_iron'] = df['bmi'] * df['iron_lvl']
    df['iron_stature'] = df['iron_lvl'] * df['stature_inch']
    df['mass_iron_ratio'] = df['mass_kg'] / df['iron_lvl']
    df['fat_chol_ratio'] = df['fat_content'] / (df['chol_good'] + df['chol_bad'])
    df['chol_good_iron_mult'] = df['chol_good'] * df['iron_lvl']
    df['lipid_enzyme3_mult'] = df['lipid_lvl'] * df['enzyme_3']
    
create_interaction_features(X_train)
create_interaction_features(X_test)

In [ ]:
    # df['bmi'] = df['mass_kg'] / ((df['stature_inch'] + 100) / 100) ** 2
    # df['waist_to_age_ratio'] = df['midsection_inch'] / 2024-df['birth']
    # df['iron_per_stature'] = df['iron_lvl'] / (df['stature_inch'] + 1e-6)
    # df['bmi_iron'] = df['bmi'] * df['iron_lvl']
    # df['iron_stature'] = df['iron_lvl'] * df['stature_inch']
    # df['mass_iron_ratio'] = df['mass_kg'] / df['iron_lvl']
    # df['fat_chol_ratio'] = df['fat_content'] / (df['chol_good'] + df['chol_bad'])
    # df['chol_good_iron'] = df['chol_good'] * df['iron_lvl']
    # df['lipid_enzyme3'] = df['lipid_lvl'] * df['enzyme_3']
    
    
    # df['Enzyme_Avg'] = df[['enzyme_1', 'enzyme_2', 'enzyme_3']].mean(axis=1)
    # df['lipid_enzyme3'] = df['lipid_lvl'] * df['enzyme_3']
    # df['iron_fat'] = df['iron_lvl'] * df['fat_content']
    # df['Waist_to_Height_Ratio'] = df['midsection_inch']/ df['stature_inch']
    # df['stature_per_cholesterol'] = df['stature_inch'] / (df['Total_Cholesterol'] + 1e-6)
    # df['iron_per_stature'] = df['iron_lvl'] / (df['stature_inch'] + 1e-6)\
    # df['Vision_Imbalance'] = (df['vision_l'] != df['vision_r'])
    # df['audio_problem'] = ((df['audio_l'] != 'N') | (df['audio_r'] != 'N')).astype(int)
    # df['bp'] = df['bp_high'] - df['bp_low']
    # df['LDL_to_HDL_Ratio'] = df['chol_bad'] / df['chol_good']
    # df['Total_Cholesterol'] = df['chol_good'] + df['chol_bad']

In [ ]:
drop=['audio_l','audio_r']

X_train=X_train.drop(drop,axis=1)
X_test=X_test.drop(drop,axis=1)

## BayesSearch : lgb, xgb, cat boost

In [ ]:
from skopt import BayesSearchCV
import os
import lightgbm as lgb
import pickle

model = lgb.LGBMClassifier(random_state=10)

# Define LightGBM parameters for GridSearchCV
params ={
    'learning_rate': (0.01,0.2),
    'max_depth': (3,8),
    'n_estimators': (200,1500),
    'subsample': (0.65,1.0),
    'num_leaves': (5,100),
    'boosting_type' : ['dart'],
    'max_bin' : (10,1000),
    
}


bayes_search_lgb = BayesSearchCV(
    model, 
    params, 
    n_iter=50,  # Specify a positive number for iterations
    scoring='roc_auc', 
    verbose=3,
    cv=3
)
bayes_search_lgb.fit(X_train, y_train)


# Extract results from the GridSearchCV output
cv_results = bayes_search_lgb.cv_results_

# Write the extracted results to the file

# Define the filename to save the results
filename = 'lgbm_bayesver2.0.txt'
# Write the extracted results to the file (create file if it doesn't exist)
with open(filename, 'a' if os.path.exists(filename) else 'x') as f:
    for mean_score, params in zip(cv_results['mean_test_score'], cv_results['params']):
        f.write(f"Params: {params}, Mean CV Score: {mean_score:.3f}\n")
        
    f.write("\nBest Parameters:\n")
    f.write(f"{bayes_search_lgb.best_params_}\n")
    f.write(f"Best Score: {bayes_search_lgb.best_score_:.3f}\n")
    f.write(f"-------------------------------------------------------------------------------------------------------------------------------------------------------------------------\n")

In [ ]:
from catboost import CatBoostClassifier
from skopt import BayesSearchCV

model = CatBoostClassifier(verbose=3)
param_space = {
    'iterations': (300,1000),          # 반복 횟수
    'depth': (3, 10),                 # 트리 깊이
    'learning_rate': (0.01, 0.1),     # 학습률
    'l2_leaf_reg': (0,5.0),           # 정규화 강도
    'border_count': (32,512),        # Border Count
}

bayes_search = BayesSearchCV(
    estimator=model,
    search_spaces=param_space,
    scoring='roc_auc',  # Precision-Recall 평가 지표 사용
    n_iter=50,                    # 탐색 반복 횟수
    cv=3,                         # 3-Fold 교차 검증
    n_jobs=-1                     # 병렬 처리
)

bayes_search.fit(X_train, y_train)
cv_results = bayes_search.cv_results_

# Define the filename to save the results
filename = 'catboost_bayesver2.0.txt'
# Write the extracted results to the file (create file if it doesn't exist)
with open(filename, 'a' if os.path.exists(filename) else 'x') as f:
    for mean_score, params in zip(cv_results['mean_test_score'], cv_results['params']):
        f.write(f"Params: {params}, Mean CV Score: {mean_score:.3f}\n")
        
    f.write("\nBest Parameters:\n")
    f.write(f"{bayes_search.best_params_}\n")
    f.write(f"Best Score: {bayes_search.best_score_:.3f}\n")
    f.write(f"-------------------------------------------------------------------------------------------------------------------------------------------------------------------------\n")


In [ ]:
from skopt import BayesSearchCV
import os
import xgboost as xgb
import pickle

# Define XGBoost model
model = xgb.XGBClassifier(random_state=10, use_label_encoder=False, eval_metric='logloss')

# Define XGBoost parameters for BayesSearchCV
params = {
    'learning_rate': (0.01, 0.2),
    'max_depth': (2, 10),
    'n_estimators': (200, 1600),
    'subsample': (0.6, 1.0),
    'colsample_bytree': (0.6, 1.0),
    'min_child_weight': (1, 10),
    'gamma': (0, 5),
    'reg_alpha': (0, 10),
    'reg_lambda': (0, 10),
}
bayes_search_xgb = BayesSearchCV(
    model,
    params,
    n_iter=50,  # Number of iterations
    scoring='roc_auc', 
    verbose=3,
    cv=3
)

# Fit the BayesSearchCV object
bayes_search_xgb.fit(X_train, y_train)

# Extract results from the GridSearchCV output
cv_results = bayes_search_xgb.cv_results_

# Define the filename to save the results
filename = 'xgb_bayesver2.0.txt'

# Write the extracted results to the file (create file if it doesn't exist)
with open(filename, 'a' if os.path.exists(filename) else 'x') as f:
    for mean_score, params in zip(cv_results['mean_test_score'], cv_results['params']):
        f.write(f"Params: {params}, Mean CV Score: {mean_score:.3f}\n")

    f.write("\nBest Parameters:\n")
    f.write(f"{bayes_search_xgb.best_params_}\n")
    f.write(f"Best Score: {bayes_search_xgb.best_score_:.3f}\n")
    f.write(f"-------------------------------------------------------------------------------------------------------------------------------------------------------------------------\n")


In [ ]:
print("cat boost")
best_model_cat=bayes_search.best_estimator_
best_params_cat=bayes_search.best_params_
print("best params : ",best_params_cat)
print()

print("lgbm boost")
best_model_lgbm=bayes_search_lgb.best_estimator_
best_params_lgbm=bayes_search.best_params_
print("best params : ",best_params_lgbm)
print()


print("xgb boost")
best_model_xgb=bayes_search_xgb.best_estimator_
best_params_xgb=bayes_search.best_params_
print("best params : ",best_params_xgb)
print()

cat_probs = best_model_cat.predict_proba(X_test)[:, 1]
lgb_probs = best_model_lgbm.predict_proba(X_test)[:, 1]
xgb_probs=best_model_xgb.predict_proba(X_test)[:, 1]
# 예측 확률 값 출력
print('best [0.53321308 0.02115297 0.03852971 ... 0.06604431 0.12076061 0.83912494]')
print('lgb ',cat_probs)
print('xgb ',lgb_probs)
print('rf  ',xgb_probs)

In [ ]:
# 제출 파일 저장
submission_df = pd.read_csv('sample_submission.csv')
submission_df.health_status = cat_probs
submission_df.to_csv('cat_submission.csv', index=False)


submission_df = pd.read_csv('sample_submission.csv')
submission_df.health_status = lgb_probs
submission_df.to_csv('lgb_submission.csv', index=False)


submission_df = pd.read_csv('sample_submission.csv')
submission_df.health_status = xgb_probs
submission_df.to_csv('xgb_submission.csv', index=False)

## Model 정의 (best prams 기준)

In [ ]:
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# catboost_model = CatBoostClassifier(
#     iterations=1000,
#     learning_rate=0.1,
#     depth=6,
#     random_seed=42
# )

# lgbm_model = LGBMClassifier(
#     n_estimators=1500,
#     learning_rate=0.2,
#     max_depth=3,
#     boosting_type= 'dart'
#     max_bin = 420
#     num_leaves= 100
#     subsample=0.7183838376338544
# )

# xgb_model = XGBClassifier(
#     n_estimators=500,
#     learning_rate=0.1,
#     max_depth=6,
#     seed=42

catboost_model = bayes_search.best_estimator_
lgbm_model = bayes_search_lgb.best_estimator_
xgb_model = bayes_search_xgb.best_estimator_

## 앙상블

### 0.5 기준 앙상블

In [ ]:
# import numpy as np  # numpy가 import되어 있는지 확인


# def ensemble_predict(xgb_probs, lgb_probs):
#     # 두 모델의 확률을 기준으로 결합
#     ensemble_probs = []
    
#     for xgb_prob, lgb_prob in zip(xgb_probs, lgb_probs,rf_probs):
#         if xgb_prob < 0.5 and lgb_prob < 0.5 and rf_probs:
#             ensemble_probs.append(min(xgb_prob, lgb_prob))
#         elif xgb_prob >= 0.5 and lgb_prob >= 0.5:
#             ensemble_probs.append(max(xgb_prob, lgb_prob))
#         else:
#             ensemble_probs.append(max(xgb_prob, lgb_prob))  # 하나만 0.5 이상일 경우
            
#     return np.array(ensemble_probs)

# # 앙상블 예측 수행
# ensemble_probs = ensemble_predict(xgb_probs, lgb_probs)

# # 앙상블 확률 출력
# print('best [0.53321308 0.02115297 0.03852971 ... 0.06604431 0.12076061 0.83912494]')
# print("Ensemble Probabilities:", ensemble_probs)

import numpy as np

def ensemble_probs(xgb_probs, lgb_probs, cat_probs):
    ensemble_probs = []
    for xgb_prob, lgb_prob, cat_prob in zip(xgb_probs, lgb_probs, cat_probs):
        # 세 개 다 0.5를 넘는 경우
        if xgb_prob > 0.5 and lgb_prob > 0.5 and cat_prob > 0.5:
            ensemble_probs.append(max(xgb_prob, lgb_prob, cat_prob))
        # 세 개 중 두 개가 0.5를 넘는 경우
        elif (xgb_prob > 0.5 and lgb_prob > 0.5) or (xgb_prob > 0.5 and cat_prob > 0.5) or (lgb_prob > 0.5 and cat_prob > 0.5):
            over_0_5 = [prob for prob in (xgb_prob, lgb_prob, cat_prob) if prob > 0.5]
            ensemble_probs.append(max(over_0_5))
        # 세 개 중 두 개가 0.5보다 아래인 경우
        elif (xgb_prob <= 0.5 and lgb_prob <= 0.5) or (xgb_prob <= 0.5 and cat_prob <= 0.5) or (lgb_prob <= 0.5 and cat_prob <= 0.5):
            under_0_5 = [prob for prob in (xgb_prob, lgb_prob, cat_prob) if prob <= 0.5]
            ensemble_probs.append(min(under_0_5))
        # 세 개 다 0.5를 넘지 못하는 경우
        else:
            ensemble_probs.append(min(xgb_prob, lgb_prob, cat_prob))
    
    return np.array(ensemble_probs)

p_ensemble_probs = ensemble_probs(xgb_probs, lgb_probs, cat_probs)
print(p_ensemble_probs)


In [ ]:
# 제출 파일 저장
submission_df = pd.read_csv('sample_submission.csv')
submission_df.health_status = p_ensemble_probs
submission_df.to_csv('p_submission.csv', index=False)

In [ ]:
catboost_model = bayes_search.best_estimator_
lgbm_model = bayes_search_lgb.best_estimator_
xgb_model = bayes_search_xgb.best_estimator_

### 스태킹

In [ ]:
from sklearn.ensemble import StackingClassifier
# 기존 모델들 및 CatBoost 추가
estimators = [
    ('xgb', xgb_model),  # XGBoost 모델
    ('lgbm', lgbm_model),  # LightGBM 모델
    ('catboost', catboost_model)  # CatBoost 모델
]

# 스태킹 모델 생성
stacking_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000),  # 메타 모델
    cv=5,  # 내부 교차 검증
    stack_method='predict_proba'  # 기본 모델의 확률값 사용
)

# 모델 학습
stacking_model.fit(X_train, y_train)

In [ ]:
stacking_model_probs = stacking_model.predict_proba(X_test)[:, 1]
print(stacking_model_probs )

In [ ]:
# 제출 파일 저장
submission_df = pd.read_csv('sample_submission.csv')
submission_df.health_status = stacking_model_probs
submission_df.to_csv('stacking_submission.csv', index=False)

## 가중치 기반 앙상블

In [ ]:
def weighted_average_predict(xgb_probs, lgb_probs, cat_probs, weights=(0.4, 0.2, 0.2)):
    # 가중치 기반 확률 평균 계산
    return weights[0] * xgb_probs + weights[1] * lgb_probs + weights[2] * cat_probs

# 예시: XGBoost, LightGBM, RandomForest 모델의 확률
weight_model_probs = weighted_average_predict(xgb_probs, lgb_probs, cat_probs, weights=(0.8, 0.1, 0.1))
print('XGB: 40%, LGB: 30%, CAT: 30%', weight_model_probs)


In [ ]:
# 제출 파일 저장
submission_df = pd.read_csv('sample_submission.csv')
submission_df.health_status = weight_model_probs
submission_df.to_csv(' weightver(8:1:1)_submission.csv', index=False)

In [ ]:
print(xgb_probs)

### 성능비교

In [ ]:
## 12.8(대시보드 기준)

# 확률 기반 앙상블 성능 : 0.85510
# 스태킹 기반 앙상블 성능 : 0.85625
# 가중치 기반 앙상블 성능 : 0.85772

In [ ]:
import numpy as np

# 앙상블 확률을 기반으로 성능 평가
def evaluate_ensemble(ensemble_probs_1, ensemble_probs_2):
    # 두 앙상블 확률 차이 계산
    auc_difference = np.abs(ensemble_probs_1 - ensemble_probs_2).mean()
    
    # 출력
    print("Average Probability Difference:", auc_difference)



# 성능 평가 수행
evaluate_ensemble(stacking_model_probs, weight_model_probs)


In [ ]:
## 0.4 0.3 0.3 : 0.05040993546242378
## 0.6 0.2 0.2 : 0.05048426036593812
## 0.5 0.2 0.3 : 0.050307805527178936
## 0.5 0.2 0.3 : 0.050307805527178936
## 0.2 0.2 0.6 : 0.050738219828466734
## 0.1 0.1 0.8 : 0.05144368735379891
## cat : 0.052624219533071045
## xgb : 0.05238306297217966
## lgb : 0.05566361855748228
## 0.8 0.1 0.1 : 0.05114085612728845